# Carga de datos

In [ ]:
import os
import warnings
import logging

# Suprimir mensajes específicos de PyTorch Lightning
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PL_DISABLE_FORK_WARNING'] = '1'

warnings.filterwarnings('ignore')

logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

In [ ]:
import pandas as pd

# Cargar datos
df = pd.read_csv('C:\\Users\\56977\\OneDrive\\Escritorio\\Tesis - copia\\Datos\\h\\2020-2026.csv', sep=';', decimal=',')

# Convertir tipos de datos
df['Barra'] = df['Barra'].astype(str)
df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
df['Valor_CLP'] = pd.to_numeric(df['Valor_CLP'], errors='coerce')

lista_barras = df['Barra'].unique()

# Crear dic de barras
df_por_barras = {}
for barra in lista_barras:
    df_barra = df[df['Barra'] == barra].copy()
    df_por_barras[barra] = df_barra

print(f"Registros totales: {len(df)}")
print(f"rango de fechas: {df['Fecha'].min()} - {df['Fecha'].max()}")
print(f"Registros por barra:")
for barra, df_barra in df_por_barras.items():
    print(f"  {barra}: {len(df_barra)}")

# Preprocesamiento Prophet

In [ ]:
from Modulos.Preprocesamiento_Prophet import test_estacionariedad
import os

ruta_test = f"C:/Users/56977/OneDrive/Escritorio/Tesis - copia/Datos/h/test_estacionariedad.csv"

if os.path.exists(ruta_test):
    df_resultados = pd.read_csv(ruta_test, index_col=0)
    print(f"Cache cargado desde: {ruta_test}")
else:
    df_resultados = test_estacionariedad(df_por_barras)
    df_resultados.to_csv(ruta_test)
    print(f"Test calculado y guardado en: {ruta_test}")

display(df_resultados)

In [ ]:
from Modulos.Preprocesamiento_Prophet import procesamiento_por_barras
import pickle

# Parámetros para el preprocesamiento
freq = 'h'
proporciones = [0.7, 0.15, 0.15]
ano_inicio = 2020
ano_fin = 2026

prepro_por_barras = procesamiento_por_barras(df_por_barras, lista_barras, freq, ano_inicio, ano_fin, proporciones, umbral_outliers=4)

# Guardar el diccionario preprocesado
with open(f'C:\\Users\\56977\\OneDrive\\Escritorio\\Tesis - copia\\Modelos_Prophet\\{freq}\\prepro_por_barras_horario.pkl', 'wb') as f:
    pickle.dump(prepro_por_barras, f)

In [ ]:
from Modulos.Preprocesamiento_Prophet import generar_estadisticas_todas_barras

# Estadísticas por barra
estadisticas_por_barra = generar_estadisticas_todas_barras(prepro_por_barras, lista_barras)

# Prophet

In [ ]:
from Modulos.Prophet_Modular import optimizacion_prophet

# Definimos los parámetros
ruta_archivo = f"Modelos_Prophet/{freq}/params_prophet_{freq}_{ano_inicio}-{ano_fin}.json"
parametros = {'grid': {'changepoint_prior_scale': [0.01, 0.02, 0.05, 0.007, 0.1, 0.2, 0.5, 0.7, 1, 2, 5, 7, 10, 20, 50, 70, 100],
                       'seasonality_prior_scale': [0.01, 0.02, 0.05, 0.007, 0.1, 0.2, 0.5, 0.7, 1, 2, 5, 7, 10],
                       'seasonality_mode': ['additive']},
            'daily_seasonality': True,
            'weekly_seasonality': True,
            'yearly_seasonality': True,
            'techo': 1.5}
modo = 'usar'

# Buscamos los mejores hiperparámetros para cada barra
metrica = 'MAE'
mejores_params_por_barra = optimizacion_prophet(prepro_por_barras, ruta_archivo, parametros, 
                                                modo, metrica, usar_cv=True, n_splits=5, hmap=False)

# Convertimos el diccionario a DataFrame
df_params = pd.DataFrame.from_dict(mejores_params_por_barra, orient='index')

# Le ponemos nombre a la columna del índice para que se vea mejor
df_params.index.name = 'Localidad'
df_params.reset_index(inplace=True)

print("TABLA DE HIPERPARÁMETROS PROPHET:")
display(df_params)

In [ ]:
from Modulos.Prophet_Modular import multi_prophet

# Parametros base para todos los modelos Prophet
params_base = {'daily_seasonality': True,
               'weekly_seasonality': True,
               'yearly_seasonality': True}
modo = 'cargar'                             # 'entrenar' o 'cargar'
carpeta_modelos = 'C:/Users/56977/OneDrive/Escritorio/Tesis - copia/Modelos_Prophet/h/modelos'

# Cargar modelos guardados
modelos, df_predicciones, df_metricas_prophet = multi_prophet(prepro_por_barras=prepro_por_barras, 
                                                              parametros=mejores_params_por_barra, 
                                                              params_base=params_base,
                                                              modo=modo,
                                                              carpeta_modelos=carpeta_modelos,
                                                              mostrar_params=True,
                                                              mostrar_graficos=False,
                                                              mostrar_componentes=False)

# Preprocesamiento Transformer

In [ ]:
# Configuración de hiperpárametros
batch_size = 128
max_prediction_length = 1
max_encoder_length = 168

# Guardamos la config y scalers
carpeta_modelos = f"Multi-Modelos_TFT/{freq}/LN/pred_{max_prediction_length}_{max_encoder_length}_cluster_con_clima"
carpeta_logs = f"Logs_TFT/{freq}/LN/pred_{max_prediction_length}_{max_encoder_length}_cluster_con_clima"

# Nombres de los experimentos
experimento_precios = f'Multi-TFT_Precios'
experimento_residuos = f'Multi-TFT_Residuos'

# Configuración TFT
tft_config = {'lr': 0.001,
              'hidden_size': 64, # Cambiar a 128?? og 32
              'heads': 4,
              'dropout': 0.1, # 0.1?? og 0.3
              'cont_size': 32, # 64?? og 8
              'patience_lr': 5}

# Early Stopping
early_stop_config = {'min_delta': 0.0001,
                     'patience': 15} # 20?? og 10

epochs = 100

In [ ]:
from Modulos.Preprocesamiento_Transformer import generar_clima, predicciones_con_clima, incluir_features_horario, holiday_binary
import contextlib, os

# Coordenadas sacadas de https://www.geodatos.net/coordenadas/chile/iquique
# Ciudades sacadas de https://www.igm.cl con miradas a https://www.coordinador.cl/wp-content/uploads/2025/11/CEN_Reporte_Energetico_SEN_Nov25.pdf
Coordenadas ={'ATACAMA': (-28.57617, -70.75938),        # Vallenar
            'CARDONES': (-27.36737, -70.33219),         # Copiapó
            'CHARRUA': (-36.82699, -73.04977),          # Concepción
            'CRUCERO': (-23.65094, -70.39752),          # Antofagasta
            'P.AZUCAR': (-29.90591, -71.25014),         # La Serena
            'P.MONTT': (-41.4693, -72.94237),           # Puerto Montt
            'QUILLOTA': (-33.036, -71.62963),           # Valparaiso
            'TARAPACA': (-20.21326, -70.15027)}         # Iquique

# Fechas
fecha_inicio = df_predicciones['ds'].min().strftime('%Y-%m-%d')
fecha_fin = df_predicciones['ds'].max().strftime('%Y-%m-%d')
carpeta_clima = f"C:\\Users\\56977\\OneDrive\\Escritorio\\Tesis - copia\\Datos\\{freq}\\Climaticos"

# Generamos los datos climáticos
datos_clima = generar_clima(fecha_inicio, fecha_fin, Coordenadas, carpeta_salida=carpeta_clima, freq=freq)

# Integrar las predicciones con los datos climáticos
dict_barras_clima = predicciones_con_clima(df_predicciones, datos_clima)

# Inluimos indicador de feriados
for barra in dict_barras_clima.keys():

    # Para mutear los prints dentro del loop:
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        df_bin = holiday_binary(dict_barras_clima[barra], prepro_por_barras[barra]['df_feriados'])
        dict_barras_clima[barra] = df_bin

# Incluimos otros features para mejorar el desempeño del Transformer
datos_para_transformer = incluir_features_horario(dict_barras_clima)

# ================== GENERACIÓN DE PARÁMETROS OBLIGATORIOS PARA PYTORCH FORECASTING ==================
# Agregamos los parámetros obligatorios para PyTorch Forecasting en todas las barras
for barra in datos_para_transformer.keys():
    # time_idx: Índice entero fundamental para que el TFT entienda la secuencia
    #datos_para_transformer[barra]['time_idx'] = datos_para_transformer[barra].index
    datos_para_transformer[barra]['time_idx'] = range(len(datos_para_transformer[barra]))
    
    # serie_id: Identificador de la serie. 
    datos_para_transformer[barra]["serie_id"] = barra

In [ ]:
# ── Features solares: hora UTC + elevación astronómica ──────────────────────
# Requiere: pip install pvlib
# Estas features son covariables FUTURAS CONOCIDAS (known_reals) porque
# la posición del sol es determinista para cualquier fecha y coordenada.

try:
    import pvlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pvlib', '-q'])
    import pvlib

import numpy as np
import pandas as pd

# ── Fechas de cambio de hora Chile Continental (para es_horario_verano) ──────
_CAMBIOS_INVIERNO = pd.to_datetime([
    '2020-04-04','2021-04-03','2022-04-02','2023-04-01',
    '2024-04-06','2025-04-05','2026-04-04',
])
_CAMBIOS_VERANO = pd.to_datetime([
    '2019-09-07','2020-09-05','2021-09-04','2022-09-10',
    '2023-09-02','2024-09-07','2025-09-06',
])
_REGIMENES = sorted(
    [(f, 1) for f in _CAMBIOS_VERANO] + [(f, 0) for f in _CAMBIOS_INVIERNO],
    key=lambda x: x[0]
)

def _es_verano(ts):
    # Devuelve 1 si el timestamp esta en UTC-3 (verano), 0 si en UTC-4 (invierno)
    ts = pd.Timestamp(ts)
    regimen = 1  # enero 2020 comienza en verano (UTC-3)
    for fecha, val in _REGIMENES:
        if fecha <= ts:
            regimen = val
        else:
            break
    return regimen


print("Agregando features solares a datos_para_transformer...")

for barra, df in datos_para_transformer.items():
    lat, lon = Coordenadas[barra]

    # ── 1. es_horario_verano ─────────────────────────────────────────────────
    df['es_horario_verano'] = df['ds'].apply(_es_verano).astype(float)

    # ── 2. Convertir hora local → UTC ────────────────────────────────────────
    ds_local = pd.to_datetime(df['ds'])
    ds_utc = ds_local.dt.tz_localize(
        'America/Santiago',
        ambiguous='NaT',        # hora ambigua (retroceso) → NaT
        nonexistent='shift_forward'  # hora inexistente (adelanto) → siguiente hora válida
    ).dt.tz_convert('UTC')

    # Rellenar NaTs (hora ambigua): usar hora anterior + 1h en UTC
    nat_mask = ds_utc.isna()
    if nat_mask.any():
        ds_utc = ds_utc.fillna(method='ffill') + pd.Timedelta(hours=1)

    hora_utc = ds_utc.dt.hour

    # Codificación cíclica de la hora UTC (ciclo de 24h)
    df['hora_utc_sin'] = np.sin(2 * np.pi * hora_utc / 24)
    df['hora_utc_cos'] = np.cos(2 * np.pi * hora_utc / 24)

    # ── 3. Elevación solar astronómica (pvlib) ───────────────────────────────
    sol = pvlib.solarposition.get_solarposition(ds_utc, lat, lon)

    df['elevacion_solar'] = sol['elevation'].values          # grados, -90 a +90
    df['cos_elevacion']   = np.cos(np.radians(sol['elevation'].values))  # 0..1

    datos_para_transformer[barra] = df

print("✓ Features solares agregadas:")
print("  - es_horario_verano  (1=UTC-3, 0=UTC-4)  → known_real")
print("  - hora_utc_sin/cos   (hora UTC cíclica)   → known_real")
print("  - elevacion_solar    (grados, -90..+90)   → known_real")
print("  - cos_elevacion      (proxy irradiancia)  → known_real")

_barra_ejemplo = list(datos_para_transformer.keys())[0]
_df_ej = datos_para_transformer[_barra_ejemplo]
print(f"\nEjemplo {_barra_ejemplo} — primeras 3 filas:")
display(_df_ej[['ds','es_horario_verano','hora_utc_sin','hora_utc_cos',
                 'elevacion_solar','cos_elevacion']].head(3))


In [ ]:
# Asumiendo que ya importaste la función
from Modulos.Preprocesamiento_Transformer import dividir_serie_temporal

# Generamos los conjuntos train, val y test para Transformer
proporciones = [0.7, 0.1495, 0.1505] # Para que me agarre los 311 días que me agarra Prophet

datasets_transformer = {}
for barra, df in datos_para_transformer.items():

    # Aseguramos el orden cronológico
    df = df.sort_values('ds').reset_index(drop=True)

    # Recalcular time_idx después del reset_index
    df['time_idx'] = range(len(df))
    df['serie_id'] = barra

    # # Aplicamos la división temporal
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        df_train, df_val, df_test = dividir_serie_temporal(df, proporciones)
    
    # Guardar en estructura organizada
    datasets_transformer[barra] = {'train': df_train,
                                   'val': df_val,
                                   'test': df_test}
    
    # Reporte rápido
    n_total = len(df)
    n_train = len(df_train)
    n_val = len(df_val)
    n_test = len(df_test)
    
    print(f"\nBarra: {barra:<5} | Train: {n_train:<5} | Val: {n_val:<4} | Test: {n_test:<4}")
    print(f"Valores: {df['y_real'].min()} a {df['y_real'].max()}")

In [ ]:
from Modulos.Preprocesamiento_Transformer import normalizar_datos, limpiar_nans

# Definimos los features y targets
feature_cols = ['trend', 'yearly', 'weekly', 'daily',                                           # Variables Prophet
                'temperatura', 'humedad', 'velocidad_viento', 'precipitacion', 'nubosidad',     # Variables climáticas
                'is_holiday',                                                                   # Indicador de feriado
                'y_lag1', 'y_lag24','y_lag168', 'resid_lag1', 'resid_lag24', 'resid_lag168',
                'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168', 'rolling_std_168',
                'rolling_mean_resid_24', 'rolling_std_resid_24', 'rolling_mean_resid_168', 'rolling_std_resid_168',
                'es_horario_verano',                                                            # Régimen horario (UTC-3/UTC-4)
                'hora_utc_sin', 'hora_utc_cos',                                                # Hora UTC cíclica
                'elevacion_solar', 'cos_elevacion']                                            # Posición solar astronómica

targets = ['y_real', 'residuo']

# Normalizamos
datasets_norm, diccionario_scalers = normalizar_datos(datasets_transformer, feature_cols, targets)

datasets_norm = limpiar_nans(lista_barras, datasets_norm)

#  Validamos las escalas, rangos y continuidad de time_idx en cada barra
for barra in lista_barras:
    print(f"\n{barra}:")
    df_train = datasets_norm[barra]['train']

    # Escalas
    print(f"  1. Escalas:")
    print(f"     y_real:  media={df_train['y_real'].mean():.3f}, std={df_train['y_real'].std():.3f}")
    print(f"     yhat:    media={df_train['yhat'].mean():.3f},   std={df_train['yhat'].std():.3f}")
    diff_std = abs(df_train['y_real'].std() - df_train['yhat'].std())

    # Continuidad time_idx
    df_val = datasets_norm[barra]['val']
    df_test = datasets_norm[barra]['test']

    train_max = df_train['time_idx'].max()
    val_min = df_val['time_idx'].min()


In [ ]:
from Modulos.Preprocesamiento_Transformer import crear_dataloaders

# Configuración de features
known_reals = ['trend', 'yearly', 'weekly', 'daily', 'is_holiday',
               'es_horario_verano',           # Régimen UTC-3 / UTC-4 (conocido de antemano)
               'hora_utc_sin', 'hora_utc_cos', # Hora UTC cíclica  (conocida de antemano)
               'elevacion_solar', 'cos_elevacion']  # Posición solar (determinista)

var_clima = ['temperatura', 'humedad', 'velocidad_viento', 'precipitacion', 'nubosidad']

unknown_reals_price = ['y_real',
                       'y_lag1', 'y_lag24', 'y_lag168',
                       'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168', 'rolling_std_168'] + var_clima

unknown_reals_resid = ['residuo',
                       'resid_lag1', 'resid_lag24', 'resid_lag168',
                       'rolling_mean_resid_24', 'rolling_std_resid_24', 'rolling_mean_resid_168', 'rolling_std_resid_168'] + var_clima

dataloaders_precios, dataloaders_residuos, datasets_precios, datasets_residuos = crear_dataloaders(lista_barras, datasets_norm,
                                                                                                   known_reals, unknown_reals_price, unknown_reals_resid,
                                                                                                   max_encoder_length, max_prediction_length, batch_size)

In [ ]:
# Diccionarios por barra
dataloaders_dict_precios = {barra: dataloaders_precios[barra] for barra in lista_barras}
dataloaders_dict_residuos = {barra: dataloaders_residuos[barra] for barra in lista_barras}

datasets_dict_precios = {barra: datasets_precios[barra]['train'] for barra in lista_barras}
datasets_dict_residuos = {barra: datasets_residuos[barra]['train'] for barra in lista_barras}

# Creamos la carpeta si no existe
os.makedirs(carpeta_modelos, exist_ok=True)
os.makedirs(carpeta_logs, exist_ok=True)

# Guardamos la configuración de features
feature_config = {'known_reals': known_reals,
                  'unknown_reals_price': unknown_reals_price,
                  'unknown_reals_resid': unknown_reals_resid,
                  'max_encoder_length': max_encoder_length,
                  'max_prediction_length': max_prediction_length,
                  'batch_size': batch_size,
                  'lista_barras': lista_barras}

# Validación preentrenamiento
print("\nVALIDACIÓN PRE-ENTRENAMIENTO")

for barra in lista_barras:
    print(f"\n{barra}:")
    
    # Verificamos el dataloader de precios
    train_loader = dataloaders_dict_precios[barra]['train']
    try:
        batch = next(iter(train_loader))
        encoder_shape = batch[0]['encoder_cont'].shape
        target_shape = batch[1][0].shape
        
        print(f"  Precios:")
        print(f"          Encoder shape: {encoder_shape}")
        print(f"          Target shape: {target_shape}")
        print(f"          Batch size: {encoder_shape[0]}")
        
        # Verificamos que no hay NaNs
        if batch[0]['encoder_cont'].isnan().any():
            print(f"      ADVERTENCIA: NaNs en encoder")
        if batch[1][0].isnan().any():
            print(f"      ADVERTENCIA: NaNs en target")
            
    except Exception as e:
        print(f"  ERROR en dataloader de precios: {e}")
        raise
    
    # Verificamos Dataloader de residuos
    train_loader_resid = dataloaders_dict_residuos[barra]['train']
    try:
        batch_resid = next(iter(train_loader_resid))
        print(f"  Residuos:")
        print(f"          Encoder shape: {batch_resid[0]['encoder_cont'].shape}")
        print(f"          Target shape: {batch_resid[1][0].shape}")
    except Exception as e:
        print(f"  ERROR en dataloader de residuos: {e}")
        raise

print(f"\nCarpeta de modelos: {carpeta_modelos}")

In [ ]:
import pickle, json

# Crear carpeta
os.makedirs(carpeta_modelos, exist_ok=True)

# 1. GUARDAR DATALOADERS (lo más importante para el cluster)
with open(f'{carpeta_modelos}/dataloaders_precios.pkl', 'wb') as f:
    pickle.dump(dataloaders_dict_precios, f)
print("✓ Dataloaders precios guardados")

with open(f'{carpeta_modelos}/dataloaders_residuos.pkl', 'wb') as f:
    pickle.dump(dataloaders_dict_residuos, f)
print("✓ Dataloaders residuos guardados")

# 2. GUARDAR SCALERS (ya lo haces, está bien)
with open(f'{carpeta_modelos}/scalers.pkl', 'wb') as f:
    pickle.dump(diccionario_scalers, f)
print("✓ Scalers guardados")

# 3. GUARDAR CONFIGURACIÓN (ya lo haces, está bien)
import numpy as np

def convert_ndarray(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_ndarray(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_ndarray(i) for i in obj]
    else:
        return obj

feature_config_serializable = convert_ndarray(feature_config)

with open(f'{carpeta_modelos}/feature_config.json', 'w') as f:
    json.dump(feature_config_serializable, f, indent=2)
print("✓ Feature config guardado")

# 4. GUARDAR CONFIGURACIÓN TFT
tft_config_total = {
    'tft_config': tft_config,
    'early_stop_config': early_stop_config,
    'epochs': epochs,
    'max_encoder_length': max_encoder_length,
    'max_prediction_length': max_prediction_length,
    'batch_size': batch_size,
    'freq': freq
}

with open(f'{carpeta_modelos}/tft_config.json', 'w') as f:
    json.dump(tft_config_total, f, indent=2)
print("✓ TFT config guardado")

In [ ]:
import pickle

# ── Rutas ──────────────────────────────────────────────────────────────────
ruta_precios  = f"{carpeta_modelos}/dataloaders_precios.pkl"
ruta_residuos = f"{carpeta_modelos}/dataloaders_residuos.pkl"

def ver_features(ruta, nombre):
    print(f"\n{'='*60}")
    print(f" {nombre}")
    print(f"{'='*60}")
    
    with open(ruta, "rb") as f:
        dl = pickle.load(f)
    
    barras = list(dl.keys())
    print(f"Barras disponibles ({len(barras)}): {barras}")
    
    for barra in barras:
        dataset = dl[barra]["train"].dataset
        print(f"\n  ── {barra} ──")
        print(f"  time_varying_known_reals:    {dataset.time_varying_known_reals}")
        print(f"  time_varying_unknown_reals:  {dataset.time_varying_unknown_reals}")
        print(f"  static_reals:                {dataset.static_reals}")
        print(f"  time_varying_known_cats:     {dataset.time_varying_known_categoricals}")
        print(f"  static_cats:                 {dataset.static_categoricals}")
        print(f"  target:                      {dataset.target}")

ver_features(ruta_precios,  "DATALOADERS PRECIOS")
ver_features(ruta_residuos, "DATALOADERS RESIDUOS")

In [ ]:
import pickle, os
ruta = carpeta_modelos  # tu carpeta
with open(os.path.join(ruta, 'datasets_norm.pkl'), 'wb') as f:
    pickle.dump(datasets_norm, f)
with open(os.path.join(ruta, 'scalers.pkl'), 'wb') as f:
    pickle.dump(diccionario_scalers, f)

# Carga de modelos LN

In [ ]:
from Modulos.TFT_Model import cargar_modelo_entrenado

modelos_precios = {}
configs_precios = {}
modelos_residuos = {}
configs_residuos = {}

for barra in lista_barras:
    modelo_precios, config_precios = cargar_modelo_entrenado(barra, "Multi-TFT_Precios", carpeta_modelos=carpeta_modelos)
    modelos_precios[barra] = modelo_precios
    configs_precios[barra] = config_precios

    modelo_residuos, config_residuos = cargar_modelo_entrenado(barra, "Multi-TFT_Residuos", carpeta_modelos=carpeta_modelos)
    modelos_residuos[barra] = modelo_residuos
    configs_residuos[barra] = config_residuos

In [ ]:
from Modulos.TFT_Model import grafico_losses

grafico_losses(lista_barras, 
               carpeta_logs, 
               experimento_precios=experimento_precios, 
               experimento_residuos=experimento_residuos)

# Evaluación LN

In [ ]:
import pickle
with open('C:\\Users\\56977\\OneDrive\\Escritorio\\Tesis - copia\\Multi-Modelos_TFT\\h\\LN\\pred_1_168_cluster_con_clima\\Resultados\\eval_TFT_LN.pkl', 'rb') as f:
    datos_ln, metricas_ln = pickle.load(f)

metricas_ln_display = metricas_ln.set_index(['Barra', 'Modelo', 'Set'])
display(metricas_ln_display)

# Analisis de Features LN

In [ ]:
import glob
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ── Cargar y combinar todos los CSV de importancia ─────────────────────────
carpeta_features = "Multi-Modelos_TFT/h/LN/pred_1_168_cluster_con_clima/Resultados/Features"
patron   = os.path.join(carpeta_features, "**", "*_importancia.csv")
archivos = glob.glob(patron, recursive=True)

dfs = []
for ruta in archivos:
    partes      = ruta.replace("\\", "/").split("/")
    experimento = partes[-3]
    barra       = partes[-2]

    df = pd.read_csv(ruta)
    df.insert(0, "Barra", barra)
    df.insert(1, "Experimento", experimento)
    dfs.append(df)

df_importancia_total = pd.concat(dfs, ignore_index=True)

# ── Heatmap + ranking por (experimento, tipo) ──────────────────────────────
for experimento in df_importancia_total["Experimento"].unique():
    for tipo in ["encoder", "decoder"]:

        subset = df_importancia_total[
            (df_importancia_total["Experimento"] == experimento) &
            (df_importancia_total["tipo"] == tipo)
        ]
        if subset.empty:
            continue

        # --- Heatmap: variable × barra ---
        pivote = subset.pivot_table(index="variable", columns="Barra", values="importancia_pct")
        pivote = pivote.loc[pivote.mean(axis=1).sort_values(ascending=False).index]  # ordenar por importancia promedio

        fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(pivote))))
        sns.heatmap(pivote, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Importancia (%)"}, ax=ax)
        ax.set_title(f"Importancia de variables ({tipo}) — {experimento}", fontsize=13, fontweight='bold')
        ax.set_xlabel("Barra")
        ax.set_ylabel("Variable")
        plt.tight_layout()
        plt.show()

        # --- Tabla resumen: ranking promedio entre barras ---
        resumen = subset.groupby("variable")["importancia_pct"].agg(["mean", "std"]).round(2)
        resumen = resumen.rename(columns={"mean": "Importancia_Media_%", "std": "Desv_Estandar_%"})
        resumen = resumen.sort_values("Importancia_Media_%", ascending=False)

        print(f"\n{'='*60}")
        print(f"  Ranking promedio — {tipo} — {experimento}")
        print(f"{'='*60}")
        display(resumen)

# Stacking Optimization LN

In [ ]:
import importlib
import Modulos.Stacking_Optimization_TFT as comp
importlib.reload(comp)
from Modulos.Stacking_Optimization_TFT import stacking_optimization

# Colores personalizados para las gráficas
mis_colores = ["#eb311c", "#e2ec1a", "#1bdd6bff", "#000000", "#2812f3", "#b300ff",]

resultados_stacking_ln, df_stacking_ln = stacking_optimization(lista_barras=lista_barras,
                                                               modelos_precios=modelos_precios, 
                                                               modelos_residuos=modelos_residuos,
                                                               dataloaders_precios=dataloaders_precios, 
                                                               dataloaders_residuos=dataloaders_residuos,
                                                               datasets_norm=datasets_norm,
                                                               diccionario_scalers=diccionario_scalers,
                                                               nombre_modelo='TFT_LN',
                                                               grid_resolution=41,
                                                               metrica_optimizacion='MAE',
                                                               colormap=None,
                                                               colores_personalizados=mis_colores,
                                                               datos_evaluacion=datos_ln,
                                                               mostrar_grafico=False)

In [ ]:
from Modulos.Stacking_Optimization_TFT import comparar_metodos_stacking

comparacion_ln, df_metodos_ln = comparar_metodos_stacking(lista_barras=lista_barras,
                                                          resultados_stacking=resultados_stacking_ln,
                                                          nombre_modelo='TFT_LN')

In [ ]:
from Modulos.Stacking_Optimization_TFT import graficar_stacking

# Solo grafica la mejor estrategia de cada barra
resultados_graficos_ln = graficar_stacking(resultados_stacking=resultados_stacking_ln,
                                           lista_barras=lista_barras,
                                           nombre_modelo='TFT_LN',
                                           zoom_dias=30,
                                           estrategias_plot=None, 
                                           metrica_mejor='MAE',
                                           conjunto_mejor='test',
                                           guardar_graficos=False,
                                           verbose=True)

df_resumen_ln = resultados_graficos_ln['df_resumen']
df_mejores_ln = resultados_graficos_ln['df_mejores']

# Cargar modelos DyT

In [ ]:
from Modulos.Parche import activar_dyt_mode, desactivar_dyt_mode
from Modulos.TFT_Model import cargar_modelo_entrenado

# Configuración
carpeta_modelos_dyt = f"Multi-Modelos_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_cluster_con_clima"
carpeta_logs_dyt = f"Logs_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_cluster_con_clima"
experimento_precios_dyt = f'Precios'
experimento_residuos_dyt = f'Residuos'

# Diccionarios para almacenar modelos
modelos_precios_dyt = {}
configs_precios_dyt = {}
modelos_residuos_dyt = {}
configs_residuos_dyt = {}

# Activamos el mode DyT
activar_dyt_mode()

try:
    for barra in lista_barras:
        
        # PRECIOS
        modelo_p, config_p = cargar_modelo_entrenado(barra=barra, 
                                                     nombre_experimento=experimento_precios_dyt, 
                                                     carpeta_modelos=carpeta_modelos_dyt)
        modelos_precios_dyt[barra] = modelo_p
        configs_precios_dyt[barra] = config_p

        # RESIDUOS
        modelo_r, config_r = cargar_modelo_entrenado(barra=barra, 
                                                     nombre_experimento=experimento_residuos_dyt, 
                                                     carpeta_modelos=carpeta_modelos_dyt)
        modelos_residuos_dyt[barra] = modelo_r
        configs_residuos_dyt[barra] = config_r
        
finally:
    # Desactivamos el modo DyT
    desactivar_dyt_mode() 

In [ ]:
from Modulos.DyT import DynamicTanh

for barra in lista_barras:
    print(f"{barra}:")
    
    for tipo, modelos_dict in [('Precios', modelos_precios_dyt), ('Residuos', modelos_residuos_dyt)]:
        modelo = modelos_dict[barra]
        
        # Contar capas DynamicTanh directamente
        dyt_real = sum(1 for m in modelo.modules() if isinstance(m, DynamicTanh))
        
        # Buscar una capa específica para inspeccionar
        sample_layer = None
        for name, module in modelo.named_modules():
            if 'norm' in name.lower() and hasattr(module, 'dyt'):
                sample_layer = name
                break
        
        print(f"   {tipo:10s}: {dyt_real} capas DynamicTanh reales")
        if sample_layer:
            print(f"              Ejemplo: {sample_layer} tiene atributo 'dyt' ✅")
    
    print()

In [ ]:
grafico_losses(lista_barras,
               carpeta_logs_dyt, 
               experimento_precios=experimento_precios_dyt, 
               experimento_residuos=experimento_residuos_dyt)

In [ ]:
import importlib
import Modulos.DyT as comp
importlib.reload(comp)
from Modulos.DyT import analizar_alpha_dyt, graficar_alphas_combinado

# Analizamos los parámetros α de DyT en el modelo de precios
alphas_por_barra = {
    barra: analizar_alpha_dyt(modelos_precios_dyt[barra], barra, verbose=False)
    for barra in modelos_precios_dyt.keys()
}

graficar_alphas_combinado(alphas_por_barra, lista_barras, titulo="Parámetros α de DyT — Precios")

# Analizamos los parámetros α de DyT en el modelo de resiudos
alphas_por_barra = {
    barra: analizar_alpha_dyt(modelos_residuos_dyt[barra], barra, verbose=False)
    for barra in modelos_residuos_dyt.keys()
}

graficar_alphas_combinado(alphas_por_barra, lista_barras, titulo="Parámetros α de DyT — Residuos")


# Evaluación DyT

In [ ]:
import pickle
with open('C:\\Users\\56977\\OneDrive\\Escritorio\\Tesis - copia\\Multi-Modelos_TFT\\h\\DyT\\pred_1_168_cluster_con_clima\\Resultados\\eval_TFT_DyT.pkl', 'rb') as f:
    datos_dyt, metricas_dyt = pickle.load(f)

metricas_dyt_display = metricas_dyt.set_index(['Barra', 'Modelo', 'Set'])
display(metricas_dyt_display)

# Analisis de Features DyT 

In [ ]:
import glob
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ── Cargar y combinar todos los CSV de importancia ─────────────────────────
carpeta_features = "Multi-Modelos_TFT/h/DyT/pred_1_168_cluster_con_clima/Resultados/Features"
patron   = os.path.join(carpeta_features, "**", "*_importancia.csv")
archivos = glob.glob(patron, recursive=True)

dfs = []
for ruta in archivos:
    partes      = ruta.replace("\\", "/").split("/")
    experimento = partes[-3]
    barra       = partes[-2]

    df = pd.read_csv(ruta)
    df.insert(0, "Barra", barra)
    df.insert(1, "Experimento", experimento)
    dfs.append(df)

df_importancia_total = pd.concat(dfs, ignore_index=True)

# ── Heatmap + ranking por (experimento, tipo) ──────────────────────────────
for experimento in df_importancia_total["Experimento"].unique():
    for tipo in ["encoder", "decoder"]:

        subset = df_importancia_total[
            (df_importancia_total["Experimento"] == experimento) &
            (df_importancia_total["tipo"] == tipo)
        ]
        if subset.empty:
            continue

        # --- Heatmap: variable × barra ---
        pivote = subset.pivot_table(index="variable", columns="Barra", values="importancia_pct")
        pivote = pivote.loc[pivote.mean(axis=1).sort_values(ascending=False).index]

        fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(pivote))))
        sns.heatmap(pivote, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Importancia (%)"}, ax=ax)
        ax.set_title(f"Importancia de variables ({tipo}) — {experimento} (DyT)", fontsize=13, fontweight='bold')
        ax.set_xlabel("Barra")
        ax.set_ylabel("Variable")
        plt.tight_layout()
        plt.show()

        # --- Tabla resumen: ranking promedio entre barras ---
        resumen = subset.groupby("variable")["importancia_pct"].agg(["mean", "std"]).round(2)
        resumen = resumen.rename(columns={"mean": "Importancia_Media_%", "std": "Desv_Estandar_%"})
        resumen = resumen.sort_values("Importancia_Media_%", ascending=False)

        print(f"\n{'='*60}")
        print(f"  Ranking promedio — {tipo} — {experimento} (DyT)")
        print(f"{'='*60}")
        display(resumen)


# Stackin Optimization DyT

In [ ]:
# Colores personalizados para las gráficas
mis_colores = ["#eb311c", "#e2ec1a", "#1bdd6bff", "#000000", "#2812f3", "#b300ff",]

resultados_stacking_dyt, df_stacking_dyt = stacking_optimization(lista_barras=lista_barras,
                                                                 modelos_precios=modelos_precios_dyt,
                                                                 modelos_residuos=modelos_residuos_dyt,
                                                                 dataloaders_precios=dataloaders_precios,
                                                                 dataloaders_residuos=dataloaders_residuos,
                                                                 datasets_norm=datasets_norm,
                                                                 diccionario_scalers=diccionario_scalers,
                                                                 nombre_modelo='TFT_DyT',
                                                                 grid_resolution=41,
                                                                 metrica_optimizacion='MAE',
                                                                 colormap=None,
                                                                 colores_personalizados=mis_colores,
                                                                 datos_evaluacion=datos_dyt)

In [ ]:
comparacion_dyt, df_metodos_dyt = comparar_metodos_stacking(lista_barras=lista_barras,
                                                          resultados_stacking=resultados_stacking_dyt,
                                                          nombre_modelo='TFT_DyT')

In [ ]:
# Graficamos los resultados
resultados_graficos_dyt = graficar_stacking(resultados_stacking=resultados_stacking_dyt,
                                           lista_barras=lista_barras,
                                           nombre_modelo='TFT_DyT',
                                           estrategias_plot=None,
                                           metrica_mejor='MAE',
                                           conjunto_mejor='test',
                                           guardar_graficos=False,
                                           verbose=True)

df_resumen_dyt = resultados_graficos_dyt['df_resumen']
df_mejores_dyt = resultados_graficos_dyt['df_mejores']

# Comparativa

In [ ]:
import importlib
import Modulos.Comparativa as comp
importlib.reload(comp)
from Modulos.Comparativa import analisis_comparativo

resultados_comparacion = analisis_comparativo(
    metricas_ln=metricas_ln,
    metricas_dyt=metricas_dyt,
    df_mejores_ln=df_mejores_ln,
    df_mejores_dyt=df_mejores_dyt,
    lista_barras=lista_barras,
    comparar='Test',
    guardar_resultados=False,
    mostrar_graficos=True,
    verbose=True
)

# Analisis de errores

In [ ]:
import importlib
import Modulos.Stacking_Optimization_TFT as comp
importlib.reload(comp)
from Modulos.Stacking_Optimization_TFT import error_distribucion, analizar_errores_barra

df_errores_ln  = error_distribucion(resultados_stacking_ln,  lista_barras, 'LN')
df_errores_dyt = error_distribucion(resultados_stacking_dyt, lista_barras, 'DyT')

In [ ]:
df_errores = analizar_errores_barra(resultados_stacking_ln, "ATACAMA", "TFT_Precios_Solo", top_n=30)

In [ ]:
df_errores_top = df_errores.sort_values(by='error_abs', ascending=False).head(50)
df_errores_top

In [ ]:
df_errores_top.sort_values(by="timestamp")

# SCP

In [ ]:
import importlib
import Modulos.Comparativa as comp
importlib.reload(comp)
from Modulos.Conformal_Prediction_Wrapper import cp_clasic_implementation, split_cp_from_calibration

# Split CP: La idea es que, despues de entrenar el modelo, se calculan los residuos en el conjunto de validación, 
# y se usa ese conjunto para calibrar el CP. Usando un cuantil de residuos q, se generan intervalos de la forma [y - q, y + q].
# Luego, se evalúa el CP en el conjunto de test para ver su desempeño real, no cambia el cuantil. 
# Intervalo fijo, ancho uniforme y no adaptativo.

cp_ln, df_cp_ln = cp_clasic_implementation(resultados_stacking=resultados_stacking_ln,
                                                   df_mejores=df_mejores_ln,
                                                   split_cp_fn=split_cp_from_calibration,
                                                   mapping='LN',
                                                   verbose=False)

cp_dyt, df_cp_dyt = cp_clasic_implementation(resultados_stacking=resultados_stacking_dyt,
                                                   df_mejores=df_mejores_dyt,
                                                   split_cp_fn=split_cp_from_calibration,
                                                   mapping='DyT',
                                                   verbose=False)
display(df_cp_ln)
display(df_cp_dyt)

In [ ]:
import importlib
import Modulos.Conformal_Prediction_Wrapper as comp
importlib.reload(comp)
from Modulos.Conformal_Prediction_Wrapper import plot_cp

# Graficamos
plot_cp(lista_barras,
        cp_ln=cp_ln,
        cp_dyt=cp_dyt,
        resultados_stacking_ln=resultados_stacking_ln,
        resultados_stacking_dyt=resultados_stacking_dyt,
        tipo_cp='Clásico') 

# CP EnbPI

In [ ]:


# EnbPI: Ensemble Bootstrap Prediction Intervals es otro método de CP que utiliza técnicas de ensemble
# y bootstrap para generar mejores intervalos de predicción, la ventaja es que son más robustos y adaptativos. 
# Entrena un RF (en este caso 100 arboles),promedia las predicciones, calcula los residuos, y luego usa bootstrap para generar 
# muestras de residuos, y luego calibra los inervalos. A diferencia del SCP, que se basa en residuos de un solo modelo, EnbPI aprovecha 
# la diversidad de múltiples modelos y muestras bootstrap para calibrar intervalos que pueden capturar mejor la 
# incertidumbre inherente a los datos y al modelo.

#for e in ['RF', 'GB', 'Ridge']:
#    print(f"\nEvaluando modelo base: {e}")
#    # LN
#    estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
#    df_enbpi_ln, cp_enbpi_ln = cp_enbpi_implementation(resultados_stacking_ln,
#                                                       lista_barras,
#                                                       estrategia_barras_ln,
#                                                       modelo_base=e,
#                                                       alpha=0.05,
#                                                       B=30)

#    print("\nRESULTADOS EnbPI LN:")
#    display(df_enbpi_ln)

#    # DyT
#    estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
#    df_enbpi_dyt, cp_enbpi_dyt = cp_enbpi_implementation(resultados_stacking_dyt,
#                                                         lista_barras,
#                                                         estrategia_barras_dyt,
#                                                         modelo_base=e,
#                                                         alpha=0.05,
#                                                         B=30)

#    print("\nRESULTADOS EnbPI DyT:")
#    display(df_enbpi_dyt)

#    # Comparar LN vs DyT
#    print("\n📈 COMPARACIÓN:")
#    comparison = pd.DataFrame({
#        "Barra": df_enbpi_ln["Barra"],
#        "Coverage_LN": df_enbpi_ln["Coverage"],
#        "Coverage_DyT": df_enbpi_dyt["Coverage"],
#        "Width_LN": df_enbpi_ln["Width"],
#        "Width_DyT": df_enbpi_dyt["Width"],
#        "MAE_LN": df_enbpi_ln["MAE"],
#        "MAE_DyT": df_enbpi_dyt["MAE"   ]
#})
#    display(comparison)

# Usar el modelo Ridge dió los mejores resultados

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import cp_enbpi_implementation

# LN
estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
df_enbpi_ln, cp_enbpi_ln = cp_enbpi_implementation(resultados_stacking_ln, 
                                                   lista_barras, 
                                                   estrategia_barras_ln,
                                                   modelo_base='Ridge',  # Cambiar a 'GB' o 'Ridge' si quieres
                                                   alpha=0.05,
                                                   B=30)

print("\nRESULTADOS EnbPI LN:")
display(df_enbpi_ln)

# DyT
estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
df_enbpi_dyt, cp_enbpi_dyt = cp_enbpi_implementation(resultados_stacking_dyt, 
                                                     lista_barras, 
                                                     estrategia_barras_dyt,
                                                     modelo_base='Ridge',
                                                     alpha=0.05,
                                                     B=30)

print("\nRESULTADOS EnbPI DyT:")
display(df_enbpi_dyt)

# Comparar LN vs DyT
print("\n📈 COMPARACIÓN:")
comparison = pd.DataFrame({
    "Barra": df_enbpi_ln["Barra"],
    "Coverage_LN": df_enbpi_ln["Coverage"],
    "Coverage_DyT": df_enbpi_dyt["Coverage"],
    "Width_LN": df_enbpi_ln["Width"],
    "Width_DyT": df_enbpi_dyt["Width"],
    "MAE_LN": df_enbpi_ln["MAE"],
    "MAE_DyT": df_enbpi_dyt["MAE"]
})
display(comparison)

In [ ]:
# Graficamos
plot_cp(lista_barras, 
        cp_enbpi_ln, 
        cp_enbpi_dyt, 
        resultados_stacking_ln, 
        resultados_stacking_dyt, 
        tipo_cp='EnbPI')

# ACP Offline

In [ ]:
import importlib
import Modulos.Comparativa as comp
importlib.reload(comp)
from Modulos.Conformal_Prediction_Wrapper import acp_offline_implementation

# Metodo de CP adaptativo, significa que cambia alpha cambia según los errores.
# Entrena el modelo una sola vez, calcula los residuos una vez y va adaptando el alpha 
# para cada predicción según la dificultad de predecir esa instancia.
# Adaptativa:
# Para cada punto del test t = 1, 2, 3,... 
# Calcular Q1, dar la predicción, observar el valor real, calcular el error, 
# actualizar alpha según el error, si el error es alto, aumentar alpha, si es bajo, 
# y luego generar el intervalo de predicción usando el nuevo alpha.  

# Probar con diferentes valores de gamma
gammas_to_test = [0.01, 0.05, 0.1]

resultados_acp = {}

for g in gammas_to_test:
    print(f"\n{'='*60}")
    print(f"🔍 EVALUANDO ACP ONLINE CON GAMMA = {g}")
    print(f"{'='*60}")
    
    # LN
    print(f"\nACP LN (gamma={g}):")
    estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_acp_off_ln, cp_acp_off_ln, alphas_acp_off_ln = acp_offline_implementation(resultados_stacking_ln, 
                                                                    lista_barras,
                                                                    estrategia_barras_ln,
                                                                    modelo_base='Ridge',
                                                                    alpha=0.05,
                                                                    gamma=g)
    display(df_acp_off_ln[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean']])
    
    # DyT
    print(f"\nACP DyT (gamma={g}):")
    estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_acp_off_dyt, cp_acp_off_dyt, alphas_acp_off_dyt = acp_offline_implementation(resultados_stacking_dyt,
                                                                       lista_barras,
                                                                       estrategia_barras_dyt,
                                                                       modelo_base='Ridge',
                                                                       alpha=0.05,
                                                                       gamma=g)
    display(df_acp_off_dyt[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean']])
    
    # Guardar resultados
    resultados_acp[g] = {'ln': (df_acp_off_ln, cp_acp_off_ln, alphas_acp_off_ln), 
                         'dyt': (df_acp_off_dyt, cp_acp_off_dyt, alphas_acp_off_dyt)}
    
    # Comparación
    print(f"\nCOMPARACIÓN LN vs DyT (gamma={g}):")
    comparison = pd.DataFrame({
        "Barra": df_acp_off_ln["Barra"],
        "Coverage_LN": df_acp_off_ln["Coverage"],
        "Coverage_DyT": df_acp_off_dyt["Coverage"],
        "Width_LN": df_acp_off_ln["Width"],
        "Width_DyT": df_acp_off_dyt["Width"],
        "MAE_LN": df_acp_off_ln["MAE"],
        "MAE_DyT": df_acp_off_dyt["MAE"],
        "Alpha_Mean_LN": df_acp_off_ln["Alpha_Mean"],
        "Alpha_Mean_DyT": df_acp_off_dyt["Alpha_Mean"]})
    display(comparison)

In [ ]:
# Graficamos
plot_cp(lista_barras, cp_acp_off_ln, cp_acp_off_dyt, resultados_stacking_ln, resultados_stacking_dyt, tipo_cp='ACP Offline')

# ACP Online

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import acp_online_implementation
import time
# Reentrena el modelo en cada iteración 
# funcionamiento es similar al offline pero con la diferencia que el modelo se reentrena
#  en cada iteración, lo que permite que el modelo se adapte a los nuevos datos y errores 
# observados.
# Adpatatividad
# Reentrena

# LN
print(f"\nACP Online LN:")
estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()

tiempo_inicio_ln = time.time()
df_acp_online_ln, cp_acp_online_ln, alphas_acp_online_ln, tiempos_ln = acp_online_implementation(resultados_stacking_ln,
                                                                                                 lista_barras,
                                                                                                 estrategia_barras_ln,
                                                                                                 modelo_base='Ridge',
                                                                                                 alpha=0.05,
                                                                                                 gamma=0.05)
tiempo_total_ln = time.time() - tiempo_inicio_ln
display(df_acp_online_ln[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean', 'Tiempo_Seg']])

# DyT
print(f"\nACP Online DyT:")
estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()

tiempo_inicio_dyt = time.time()
df_acp_online_dyt, cp_acp_online_dyt, alphas_acp_online_dyt, tiempos_dyt = acp_online_implementation(resultados_stacking_dyt,
                                                                                                     lista_barras,
                                                                                                     estrategia_barras_dyt,
                                                                                                     modelo_base='Ridge',
                                                                                                     alpha=0.05,
                                                                                                     gamma=0.05)
tiempo_total_dyt = time.time() - tiempo_inicio_dyt
display(df_acp_online_dyt[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean', 'Tiempo_Seg']])

# Comparación LN vs DyT
print(f"\nCOMPARACIÓN LN vs DyT (ACP Online):")
comparison = pd.DataFrame({
    "Barra": df_acp_online_ln["Barra"],
    "Coverage_LN": df_acp_online_ln["Coverage"],
    "Coverage_DyT": df_acp_online_dyt["Coverage"],
    "Width_LN": df_acp_online_ln["Width"],
    "Width_DyT": df_acp_online_dyt["Width"],
    "MAE_LN": df_acp_online_ln["MAE"],
    "MAE_DyT": df_acp_online_dyt["MAE"],
    "RMSE_LN": df_acp_online_ln["RMSE"],
    "RMSE_DyT": df_acp_online_dyt["RMSE"],
    "Tiempo_LN": df_acp_online_ln["Tiempo_Seg"],
    "Tiempo_DyT": df_acp_online_dyt["Tiempo_Seg"]
})
display(comparison)

print("\nTIEMPOS DE EJECUCIÓN:")
print(f"  ACP Online LN:  {tiempo_total_ln:.2f}s")
print(f"  ACP Online DyT: {tiempo_total_dyt:.2f}s")
print(f"  TOTAL:          {tiempo_total_ln + tiempo_total_dyt:.2f}s")

In [ ]:
# Graficamos
plot_cp(lista_barras, cp_acp_online_ln, cp_acp_online_dyt, resultados_stacking_ln, resultados_stacking_dyt, tipo_cp='ACP Online')

# Comparación CP

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import comparativa_metodos_cp

todos_metodos = {
    'CP Clásico': {'ln': df_cp_ln, 'dyt': df_cp_dyt, 'cp_ln': cp_ln, 'cp_dyt': cp_dyt,
                   'color': '#FFD93D', 'marker': 'D'},
    'EnbPI':      {'ln': df_enbpi_ln, 'dyt': df_enbpi_dyt, 'cp_ln': cp_enbpi_ln, 'cp_dyt': cp_enbpi_dyt,
                   'color': '#FF6B6B', 'marker': 'o'},
    'ACP Offline':{'ln': df_acp_off_ln, 'dyt': df_acp_off_dyt, 'cp_ln': cp_acp_off_ln, 'cp_dyt': cp_acp_off_dyt,
                   'color': '#4ECDC4', 'marker': 's'},
    'ACP Online': {'ln': df_acp_online_ln, 'dyt': df_acp_online_dyt, 'cp_ln': cp_acp_online_ln, 'cp_dyt': cp_acp_online_dyt,
                   'color': '#45B7D1', 'marker': '^'},
}

df_general, df_por_barra = comparativa_metodos_cp(todos_metodos, lista_barras)


# Alternativo

In [ ]:
# Cargar el módulo
import importlib
import sys
sys.path.append(r'C:\Users\56977\OneDrive\Escritorio\Tesis - copia')

from Modulos.Uncertainty_Comparison import compare_uncertainty_methods, plot_comparison
importlib.reload(sys.modules.get('Modulos.Uncertainty_Comparison'))

# Ejecutar comparación en ATACAMA
results = compare_uncertainty_methods(
    barra='ATACAMA',
    resultados_stacking=resultados_stacking_ln,  # Del stacking LN
    df_mejores=df_mejores_ln,
    mapping='LN',
    alpha=0.05,
    epochs=150,
    verbose=True
)

# Visualizar
if results is not None:
    plot_comparison(results, 
                   save_path='comparacion_incertidumbre_ATACAMA.png',
                   show=True)
    
    # Mostrar tabla resumida
    print("\n" + "="*70)
    print("TABLA RESUMIDA")
    print("="*70)
    display(results["df_summary"])

# LW ACP

In [ ]:
import importlib
import Modulos.Conformal_Prediction_Wrapper as comp
importlib.reload(comp)
from Modulos.Conformal_Prediction_Wrapper import lw_split_cp_implementation, lw_acp_online_implementation, plot_cp_por_hora

# Sistema que agrega peso según la hora del día, para que los errores de ciertas horas
# tengan más peso en el cálculo del cuantil, agregado a SCP.

# LW-Split CP
print("=" * 70)
print("LW-SPLIT CP")
print("=" * 70)
cp_lw_split_ln, df_lw_split_ln = lw_split_cp_implementation(
    resultados_stacking_ln, df_mejores_ln, alpha=0.05, mapping='LN', verbose=False)
display(df_lw_split_ln)

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import plot_cp

plot_cp(
    lista_barras=lista_barras,
    cp_ln=cp_lw_split_ln,
    cp_dyt=cp_lw_split_ln,
    resultados_stacking_ln=resultados_stacking_ln,
    resultados_stacking_dyt=resultados_stacking_ln,
    tipo_cp='LW-SPLIT CP',
    zoom_dias=30,
)

In [ ]:
# Mismo sistema de pesos por hora, pero aplicado a ACP Online, es decir, 
# el cuantil se calcula con pesos según la hora del día.

# LW-ACP Online (gamma=0.01)
print("\n" + "=" * 70)
print("LW-ACP ONLINE (gamma=0.01)")
print("=" * 70)
df_lw_acp_ln, cp_lw_acp_ln, alphas_lw_acp_ln = lw_acp_online_implementation(
    resultados_stacking_ln, df_mejores_ln, alpha=0.05, gamma=0.01, mapping='LN', verbose=False)
display(df_lw_acp_ln)

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import plot_cp

plot_cp(
    lista_barras=lista_barras,
    cp_ln=cp_lw_acp_ln,
    cp_dyt=cp_lw_acp_ln,
    resultados_stacking_ln=resultados_stacking_ln,
    resultados_stacking_dyt=resultados_stacking_ln,
    tipo_cp='LW-ACP Online',
    zoom_dias=30,
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plotear la distribución de errores por hora para estudiar posibles patrones de 
# error según la hora del día.

barra = "ATACAMA"

# Buscar estrategia correcta automáticamente
mapping_ln = {
    "Prophet Solo": "Prophet_Solo",
    "TFT_LN Precios Solo": "TFT_Precios_Solo",
    "Prophet+TFT_LN Precios": "Prophet_TFT_Precios",
    "TFT Residuos Solo": "TFT_Residuos_Solo",
    "Prophet + Residuos (Directo)": "Prophet_Residuos_Directo",
    "Prophet + Residuos (Opt)": "Prophet_Residuos_Opt",
    "(Prophet + Residuos) + TFT_LN Precios": "Prophet_Residuos_Precios"
}
best_pretty = df_mejores_ln[df_mejores_ln['Barra'] == barra]['Mejor Estrategia'].values[0]
best_key    = mapping_ln[best_pretty]
print(f"{barra} → estrategia: {best_pretty} ({best_key})")

y_val      = np.asarray(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
yhat_val   = np.asarray(resultados_stacking_ln[barra][best_key]['prediccion_val'])
fechas_val = resultados_stacking_ln[barra]['_datos_split']['fechas_val']

errores   = np.abs(y_val - yhat_val)
horas_val = pd.DatetimeIndex(fechas_val).hour

# Figura: 24 subplots (4 filas × 6 columnas)
fig, axes = plt.subplots(4, 6, figsize=(20, 13))
axes = axes.flatten()
horas_transicion = {7, 8, 19, 20, 21}

for h in range(24):
    ax    = axes[h]
    err_h = errores[horas_val == h]
    color = '#E53935' if h in horas_transicion else '#1E88E5'

    ax.hist(err_h, bins=40, color=color, alpha=0.75, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', lw=1.2, ls='--')
    ax.axvline(np.median(err_h), color='gold', lw=1.5, ls='-')

    mae_h = np.mean(np.abs(err_h))
    p95_h = np.percentile(np.abs(err_h), 95)

    ax.set_title(f'h={h:02d}  n={len(err_h)}', fontsize=9, fontweight='bold',
                 color='#B71C1C' if h in horas_transicion else 'black')
    ax.set_xlabel('Error (USD/MWh)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    ax.text(0.97, 0.95, f'MAE={mae_h:.1f}\nP95={p95_h:.1f}',
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

fig.suptitle(f'Distribución de errores por hora — {barra} (set Val)\n'
             f'Rojo = horas de transición solar (7-8h, 19-21h)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# Figura 1: Width y coverage por hora — los 5 métodos
metodos_comparacion = {
    "Split CP":      cp_ln,
    "LW-Split CP":   cp_lw_split_ln,
    "ACP Online":    cp_acp_online_ln,
    "LW-ACP Online": cp_lw_acp_ln,
    "LW-SCP":        cp_lw_split_ln,
}
plot_cp_por_hora(metodos_comparacion, resultados_stacking_ln, barra="ATACAMA")


In [ ]:
import numpy as np
import pandas as pd

res = cp_lw_acp_ln["ATACAMA"]
y_test = res["y_test"]
lower  = res["lower"]
upper  = res["upper"]
horas  = res["horas_test"]

width_por_hora = pd.Series({
    h: np.mean(upper[horas == h] - lower[horas == h])
    for h in range(24)
}, name="Width_LW_ACP")

print(width_por_hora.round(1).to_string())
print(f"\nPromedio global: {width_por_hora.mean():.1f} USD/MWh")
print(f"Mínimo (hora {width_por_hora.idxmin()}h): {width_por_hora.min():.1f} USD/MWh")
print(f"Máximo (hora {width_por_hora.idxmax()}h): {width_por_hora.max():.1f} USD/MWh")


# Análisis: Errores extremos y cambios de hora en Chile

**Hipótesis:** Los errores más grandes del TFT ocurren en fechas cercanas a los cambios de hora en Chile, porque el modelo aprendió el patrón de transición solar según el reloj y se desajusta cuando el reloj adelanta o atrasa 1 hora.

**Cambios de hora confirmados (Chile Continental, excluye Magallanes):**
- **Inicio invierno (UTC-4, reloj atrasa 1 h):** primer sábado de abril cada año.
- **Inicio verano (UTC-3, reloj adelanta 1 h):** primer sábado de septiembre — excepción 2022: 10-sep por decreto especial (DS N°224/2022).

Fuentes: directemar.cl, elmostrador.cl, decreto DS N°224/2022 Ministerio del Interior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Fechas de cambio de hora Chile Continental (2020-2026) ──
FECHAS_INVIERNO = pd.to_datetime([
    '2020-04-04', '2021-04-03', '2022-04-02', '2023-04-01',
    '2024-04-06', '2025-04-05', '2026-04-04',
])
FECHAS_VERANO = pd.to_datetime([
    '2019-09-07', '2020-09-05', '2021-09-04', '2022-09-10',
    '2023-09-02', '2024-09-07', '2025-09-06',
])
FECHAS_CAMBIO_HORA = FECHAS_INVIERNO.append(FECHAS_VERANO).sort_values()

TIPO_CAMBIO = {}
for _f in FECHAS_INVIERNO: TIPO_CAMBIO[str(_f.date())] = 'invierno'
for _f in FECHAS_VERANO:
    TIPO_CAMBIO[str(_f.date())] = 'verano*' if str(_f.date()) == '2022-09-10' else 'verano'

def _dias_min(ts):
    return int(np.abs((FECHAS_CAMBIO_HORA - pd.Timestamp(ts)).days).min())

def _cambio_cercano(ts):
    diffs = np.abs((FECHAS_CAMBIO_HORA - pd.Timestamp(ts)).days)
    return FECHAS_CAMBIO_HORA[diffs.argmin()]

def _get_key(resultados, barra, nombre_buscado):
    """Resuelve nombre legible → clave real del dict de resultados."""
    for key, val in resultados[barra].items():
        if not key.startswith('_') and isinstance(val, dict):
            if val.get('nombre') == nombre_buscado:
                return key
    # Fallback: buscar por MAE en validación (mismo criterio que error_distribucion)
    y_val = np.array(resultados[barra]['_datos_split']['y_real_val'])
    claves = [k for k in resultados[barra] if not k.startswith('_')]
    maes   = {k: np.abs(y_val - np.array(resultados[barra][k]['prediccion_val'])).mean()
              for k in claves}
    return min(maes, key=maes.get)

print(f"Fechas de cambio de hora cargadas: {len(FECHAS_CAMBIO_HORA)}")
display(pd.DataFrame({'Fecha': FECHAS_CAMBIO_HORA,
                      'Tipo':  [TIPO_CAMBIO[str(f.date())] for f in FECHAS_CAMBIO_HORA]}))

In [ ]:
# ── Paso 1: Top-20 errores por barra con indicador de cambio de hora ──
VENTANA_DIAS = 7

filas_top20 = []
for barra in lista_barras:
    nombre_est = df_mejores_ln.loc[df_mejores_ln['Barra'] == barra, 'Mejor Estrategia'].values[0]
    est_key    = _get_key(resultados_stacking_ln, barra, nombre_est)

    y_val      = np.array(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
    yhat_val   = np.array(resultados_stacking_ln[barra][est_key]['prediccion_val'])
    fechas_val = pd.to_datetime(resultados_stacking_ln[barra]['_datos_split']['fechas_val'])

    errores  = np.abs(y_val - yhat_val)
    df_barra = pd.DataFrame({
        'Barra': barra, 'Estrategia': nombre_est,
        'timestamp': fechas_val, 'y_real': y_val, 'y_pred': yhat_val, 'error_abs': errores,
    }).nlargest(20, 'error_abs').reset_index(drop=True)
    df_barra['rank'] = df_barra.index + 1

    df_barra['dias_al_cambio'] = df_barra['timestamp'].apply(_dias_min)
    df_barra['cambio_cercano'] = df_barra['timestamp'].apply(_cambio_cercano)
    df_barra['tipo_cambio']    = df_barra['cambio_cercano'].apply(
        lambda d: TIPO_CAMBIO.get(str(d.date()), '?'))
    df_barra['en_ventana_1d']  = df_barra['dias_al_cambio'] <= 1
    df_barra['en_ventana_7d']  = df_barra['dias_al_cambio'] <= VENTANA_DIAS
    filas_top20.append(df_barra)

df_top20_all = pd.concat(filas_top20, ignore_index=True)

resumen = df_top20_all.groupby('Barra').agg(
    n_en_1d=('en_ventana_1d', 'sum'),
    pct_1d =('en_ventana_1d', lambda x: f"{x.mean()*100:.0f}%"),
    n_en_7d=('en_ventana_7d', 'sum'),
    pct_7d =('en_ventana_7d', lambda x: f"{x.mean()*100:.0f}%"),
).reset_index()

print("=== Cuántos del top-20 caen en ventana de cambio de hora (por barra) ===")
display(resumen)
print("\n=== Detalle top-20 (todas las barras) ===")
cols = ['Barra', 'rank', 'timestamp', 'error_abs', 'dias_al_cambio', 'cambio_cercano',
        'tipo_cambio', 'en_ventana_7d']
display(df_top20_all[cols].sort_values(['Barra', 'rank']))

In [ ]:
# ── Paso 2: MAE por grupo temporal (±1d, ±2-7d, resto del año) ──
filas_mae = []
for barra in lista_barras:
    nombre_est = df_mejores_ln.loc[df_mejores_ln['Barra'] == barra, 'Mejor Estrategia'].values[0]
    est_key    = _get_key(resultados_stacking_ln, barra, nombre_est)

    y_val      = np.array(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
    yhat_val   = np.array(resultados_stacking_ln[barra][est_key]['prediccion_val'])
    fechas_val = pd.to_datetime(resultados_stacking_ln[barra]['_datos_split']['fechas_val'])
    errores    = np.abs(y_val - yhat_val)
    dias       = np.array([_dias_min(t) for t in fechas_val])

    m1   = dias <= 1
    m7   = (dias > 1) & (dias <= 7)
    mres = dias > 7

    mae_1d   = errores[m1].mean()   if m1.any()   else np.nan
    mae_7d   = errores[m7].mean()   if m7.any()   else np.nan
    mae_rest = errores[mres].mean() if mres.any() else np.nan

    filas_mae.append({
        'Barra':           barra,
        'Estrategia':      nombre_est,
        'MAE ±1d':         mae_1d,
        'n ±1d':           int(m1.sum()),
        'MAE ±2-7d':       mae_7d,
        'n ±2-7d':         int(m7.sum()),
        'MAE resto':       mae_rest,
        'n resto':         int(mres.sum()),
        'Ratio ±1d/resto': round(mae_1d / mae_rest, 2) if not np.isnan(mae_1d) else np.nan,
    })

df_mae = pd.DataFrame(filas_mae).round(3)
prom   = df_mae[['MAE ±1d', 'MAE ±2-7d', 'MAE resto', 'Ratio ±1d/resto']].mean().round(3)
df_mae_display = pd.concat(
    [df_mae, pd.DataFrame([{'Barra': 'PROMEDIO', **prom.to_dict()}])], ignore_index=True)

print("=== MAE por grupo temporal — modelo LN ===")
display(df_mae_display[['Barra', 'Estrategia', 'MAE ±1d', 'n ±1d',
                          'MAE ±2-7d', 'n ±2-7d', 'MAE resto', 'n resto', 'Ratio ±1d/resto']])

In [ ]:
# ── Paso 3A: Serie completa + zooms ±14d alrededor de cada cambio de hora ──
for barra in lista_barras:
    nombre_est = df_mejores_ln.loc[df_mejores_ln['Barra'] == barra, 'Mejor Estrategia'].values[0]
    est_key    = _get_key(resultados_stacking_ln, barra, nombre_est)

    y_val      = np.array(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
    yhat_val   = np.array(resultados_stacking_ln[barra][est_key]['prediccion_val'])
    fechas_val = pd.to_datetime(resultados_stacking_ln[barra]['_datos_split']['fechas_val'])
    errores    = np.abs(y_val - yhat_val)

    f_min, f_max = fechas_val.min(), fechas_val.max()
    cambios_prox  = [f for f in FECHAS_CAMBIO_HORA
                     if (f_min - pd.Timedelta(days=14)) <= f <= (f_max + pd.Timedelta(days=14))]

    if not cambios_prox:
        print(f"{barra}: sin cambios de hora en el período de validación.")
        continue

    # ── Panel A1: serie completa ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(18, 4))
    ax.plot(fechas_val, errores, color='steelblue', lw=0.5, alpha=0.7, label='Error absoluto')
    ax.axhline(errores.mean(), color='gray', lw=1, ls='--', label=f'MAE={errores.mean():.2f}')
    added = set()
    for fecha in cambios_prox:
        tipo  = TIPO_CAMBIO.get(str(fecha.date()), 'cambio')
        color = '#e74c3c' if 'verano' in tipo else '#2980b9'
        lbl   = f'Cambio {tipo}' if tipo not in added else '_nolegend_'
        added.add(tipo)
        ax.axvline(fecha, color=color, lw=1.5, ls=':', alpha=0.85, label=lbl)
    ax.set_title(f'{barra} [{nombre_est}] — Error absoluto (validación) con cambios de hora',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Error absoluto (USD/MWh)')
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()

    # ── Panel A2: zooms ±14 días por cada cambio dentro del período val ──────
    cambios_en_val = [f for f in FECHAS_CAMBIO_HORA if f_min <= f <= f_max]
    if not cambios_en_val:
        continue

    n = len(cambios_en_val)
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 4 * nrows), sharey=True,
                             squeeze=False)
    axes_flat = axes.flatten()

    mae_global = errores.mean()
    for i, fecha in enumerate(cambios_en_val):
        ax = axes_flat[i]
        mask = ((fechas_val >= fecha - pd.Timedelta(days=14)) &
                (fechas_val <= fecha + pd.Timedelta(days=14)))
        tipo  = TIPO_CAMBIO.get(str(fecha.date()), 'cambio')
        color = '#e74c3c' if 'verano' in tipo else '#2980b9'
        mae_ventana = errores[mask].mean() if mask.any() else np.nan

        ax.plot(fechas_val[mask], errores[mask], color='steelblue', lw=0.9, alpha=0.85)
        ax.axhline(mae_global,   color='gray',  lw=1,   ls='--', alpha=0.6,
                   label=f'MAE global={mae_global:.2f}')
        ax.axhline(mae_ventana,  color=color,   lw=1.2, ls='-.',  alpha=0.75,
                   label=f'MAE ventana={mae_ventana:.2f}')
        ax.axvline(fecha, color=color, lw=2, ls=':', label=f'{tipo} ({fecha.date()})')
        ax.set_title(f'Zoom ±14d | {fecha.date()} ({tipo})', fontsize=9, fontweight='bold')
        ax.set_xlabel('Fecha')
        ax.tick_params(axis='x', rotation=30, labelsize=7)
        if i % ncols == 0:
            ax.set_ylabel('Error absoluto (USD/MWh)')
        ax.legend(fontsize=7)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'{barra} — Zoom ±14 días alrededor de cada cambio de hora (validación)',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Paso 3B: Violinplot de errores por grupo temporal ──
n_cols = 4
n_rows = (len(lista_barras) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

COLORES = ['#e74c3c', '#f39c12', '#2980b9']
LABELS  = ['±1d\ncambio hora', '±2–7d\ncambio hora', 'Resto\ndel año']

for i, barra in enumerate(lista_barras):
    ax = axes[i]
    nombre_est = df_mejores_ln.loc[df_mejores_ln['Barra'] == barra, 'Mejor Estrategia'].values[0]
    est_key    = _get_key(resultados_stacking_ln, barra, nombre_est)

    y_val      = np.array(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
    yhat_val   = np.array(resultados_stacking_ln[barra][est_key]['prediccion_val'])
    fechas_val = pd.to_datetime(resultados_stacking_ln[barra]['_datos_split']['fechas_val'])
    errores    = np.abs(y_val - yhat_val)
    dias       = np.array([_dias_min(t) for t in fechas_val])

    grupos = [errores[dias <= 1], errores[(dias > 1) & (dias <= 7)], errores[dias > 7]]
    datos  = [g for g in grupos if len(g) > 5]
    labels = [l for g, l in zip(grupos, LABELS) if len(g) > 5]
    cols   = [c for g, c in zip(grupos, COLORES) if len(g) > 5]

    if not datos:
        ax.set_visible(False)
        continue

    vp = ax.violinplot(datos, showmedians=True, showextrema=False)
    for body, color in zip(vp['bodies'], cols):
        body.set_facecolor(color)
        body.set_alpha(0.65)
    vp['cmedians'].set_color('black')
    vp['cmedians'].set_linewidth(1.5)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Error absoluto (USD/MWh)', fontsize=8)
    ax.set_title(f'{barra}', fontsize=10, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de errores absolutos por grupo temporal — LN (validación)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Paso 4: Gráficos ±48h para casos del top-20 cercanos a cambio de hora ──
casos = df_top20_all[df_top20_all['en_ventana_7d']].copy()
print(f"Casos del top-20 dentro de ventana ±7d: {len(casos)} "
      f"(de {len(df_top20_all)} totales en {df_top20_all['Barra'].nunique()} barras)")
display(casos[['Barra', 'rank', 'timestamp', 'error_abs',
               'dias_al_cambio', 'cambio_cercano', 'tipo_cambio']])

for _, fila in casos.iterrows():
    barra      = fila['Barra']
    nombre_est = fila['Estrategia']
    est_key    = _get_key(resultados_stacking_ln, barra, nombre_est)
    t_error    = fila['timestamp']
    t_cambio   = fila['cambio_cercano']
    tipo       = fila['tipo_cambio']

    y_val      = np.array(resultados_stacking_ln[barra]['_datos_split']['y_real_val'])
    yhat_val   = np.array(resultados_stacking_ln[barra][est_key]['prediccion_val'])
    fechas_val = pd.to_datetime(resultados_stacking_ln[barra]['_datos_split']['fechas_val'])

    mask = ((fechas_val >= t_error - pd.Timedelta(hours=48)) &
            (fechas_val <= t_error + pd.Timedelta(hours=48)))

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(fechas_val[mask], y_val[mask],    label='Real',       color='#2c3e50', lw=1.8)
    ax.plot(fechas_val[mask], yhat_val[mask], label='Predicción', color='#e74c3c', lw=1.8, ls='--')
    ax.axvline(t_error,  color='#f39c12', lw=2.5, ls=':',
               label=f'Error pico #{int(fila["rank"])} ({fila["error_abs"]:.1f} USD/MWh)')
    ax.axvline(t_cambio, color='#27ae60', lw=2.5, ls='--',
               label=f'Cambio {tipo} ({t_cambio.date()}, Δ{int(fila["dias_al_cambio"])}d)')

    ax.set_title(f'{barra} [{nombre_est}] — Ventana ±48h | Error #{int(fila["rank"])} '
                 f'| {t_error}', fontweight='bold')
    ax.set_xlabel('Fecha / Hora')
    ax.set_ylabel('CMg (USD/MWh)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()